<a href="https://colab.research.google.com/github/4cekay/NLP-Playground/blob/off-project/Medical_Simplifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Setup**

In [1]:
!pip install -q huggingface_hub gradio transformers torch accelerate

In [2]:
import os
import torch
import torch.nn as nn
import gradio as gr
from transformers import AutoTokenizer, AutoConfig, AutoModel, PreTrainedModel, pipeline
from huggingface_hub import InferenceClient

# **BioMistral**

Loading a model for obtaining medical context.

In [3]:
pipe = pipeline("text-generation", model="BioMistral/BioMistral-7B-SLERP")
messages = [
    {"role": "user", "content": "Who are you?"},
]
pipe(messages)

config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/22.8k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

...

tokenizer.model: reconstructing file:   0%|          |  0.00B /  493kB            

tokenizer.model: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer TokenizersBackend. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


[{'generated_text': [{'role': 'user', 'content': 'Who are you?'},
   {'role': 'assistant',
    'content': "I'm Mistral, a language model trained by the Mistral AI team."}]}]

**1.)** Test the model with a basic question prompt inquiring about a medical condition (similar to how a user would).

In [4]:
prompt_1 = [
    {"role": "user",
     "content": "What's a pneumothorax?"
     }
]

med_output = pipe(prompt_1)

In [25]:
# extract the final generated answer from the medical model
complex_response = med_output[0]['generated_text'][1]['content']
print(complex_response)

A pneumothorax is a medical condition that occurs when air gets trapped inside the space between the lung and chest wall. This can cause the lung to collapse or become smaller, which can lead to breathing difficulties and even life-threatening complications. Pneumothorax can be caused by a variety of factors, including lung injuries, certain medical conditions, and the use of certain medications. It is important to seek medical attention if you suspect you may have a pneumothorax, as prompt treatment can help prevent serious complications.


In [38]:
# testing the medical model's tonal capabilitiies by asking it to simplify directly
prompt_2 = [
    {"role": "user",
     "content": "Can you explain a pneumothorax to me in simple terms?"
     }
]

med_output = pipe(prompt_2)
print(med_output[0]['generated_text'][1]['content'])

[transformers] Both `max_new_tokens` (=256) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Yes, I can explain a pneumothorax to you in simple terms. A pneumothorax is a medical condition in which there is an abnormal collection of air in the space between the lungs and the chest wall. This can happen for various reasons, such as a sudden increase in air pressure in the lungs, a puncture wound in the chest, or a medical procedure that involves inserting a tube into the lungs. When this happens, the air can cause the lungs to collapse, which can lead to difficulty breathing and other complications. It's important to seek medical attention if you experience symptoms of a pneumothorax, such as chest pain, shortness of breath, or difficulty speaking, as it can be a life-threatening condition.


# **GPT2**

Like the lab example, except I'm spoonfeeding the model the relevant context generated by the medical model (I don't have medical documents to retrieve from sorry)

In [ ]:
rag_model = pipeline('text-generation', model='gpt2')

# **Kimi-K3**

(HF Inference Providers)

Asking a very large agent model (wayyyyy too large for this task, i just picked the one with most downloads lol) to create a simplified breakdown of the medical model's output to see its simplification reasoning.

In [23]:
try:
  from google.colab import userdata
  HF_TOKEN = userdata.get("HF_TOKEN")
except ImportError:
  HF_TOKEN = os.environ.get("HF_TOKEN")

In [24]:
model = "moonshotai/Kimi-K3"

client = InferenceClient(model, token=HF_TOKEN)

In [30]:
context = complex_response

query = prompt_1[0]['content']

messages = [
    {
      "role": "system",
      "content": f"Use the following scientific context: '{context}', to write a simplified response to the query in layman's terms for easier comprehension."
    },
    {
      "role": "user",
      "content": query
    }
]

In [34]:
response = client.chat.completions.create(
    model=model,
    messages=messages,
)

result = response.choices[0].message.content
print("Kimi-K3 Simplified Response:", result)

Kimi-K3 Simplified Response: ChatCompletionOutputMessage(role='assistant', content='A pneumothorax is basically a "collapsed lung." Here\'s what happens in simple terms:\n\nYour lungs sit inside your chest, and there\'s a thin space between each lung and the chest wall. Normally, that space is airtight. But sometimes air leaks into it and gets trapped there. When that happens, the trapped air pushes on the lung, making it shrink or collapse—kind of like a balloon losing air.\n\nThis can make it hard to breathe, and in serious cases, it can be dangerous or even life-threatening.\n\n**What can cause it?**\n- An injury to the chest (like from a car accident or a broken rib)\n- Certain lung conditions or diseases\n- Some medications\n\n**The key takeaway:** If you think you might have a collapsed lung—for example, if you suddenly have chest pain or trouble breathing—see a doctor right away. Getting treated quickly can prevent serious problems.', reasoning=None, tool_call_id=None, tool_call

In [35]:
result = response.choices[0].message.content
print("Kimi-K3 Simplified Response:", result)

Kimi-K3 Simplified Response: A pneumothorax is basically a "collapsed lung." Here's what happens in simple terms:

Your lungs sit inside your chest, and there's a thin space between each lung and the chest wall. Normally, that space is airtight. But sometimes air leaks into it and gets trapped there. When that happens, the trapped air pushes on the lung, making it shrink or collapse—kind of like a balloon losing air.

This can make it hard to breathe, and in serious cases, it can be dangerous or even life-threatening.

**What can cause it?**
- An injury to the chest (like from a car accident or a broken rib)
- Certain lung conditions or diseases
- Some medications

**The key takeaway:** If you think you might have a collapsed lung—for example, if you suddenly have chest pain or trouble breathing—see a doctor right away. Getting treated quickly can prevent serious problems.


In [37]:
print(response.choices[0].message.reasoning_content)

The user is asking a simple question: "What's a pneumothorax?" They've provided scientific context and asked me to write a simplified response in layman's terms for easier comprehension.

So my task is to explain pneumothorax in simple, everyday language that anyone can understand. Let me break down the medical jargon:

- "Pneumothorax" = collapsed lung (air where it shouldn't be)
- "Air gets trapped inside the space between the lung and chest wall" = air leaks into the space around the lung
- "Lung to collapse or become smaller" = the lung can't inflate properly
- "Breathing difficulties and even life-threatening complications" = trouble breathing, can be dangerous
- "Caused by lung injuries, certain medical conditions, and the use of certain medications" = can happen from injuries, health problems, or some medicines
- "Seek medical attention" = see a doctor right away

Let me write this in simple, friendly language:

A pneumothorax is sometimes called a "collapsed lung." It happens w